<a href="https://colab.research.google.com/github/diwakarg05/Neural-Network/blob/main/six_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download dataset
path = kagglehub.dataset_download("hriitk2026/plant-leaf")

print("Dataset Path:", path)

100%|██████████| 1.27G/1.27G [00:09<00:00, 144MB/s]

Extracting files...


Dataset Path: /root/.cache/kagglehub/datasets/hriitk2026/plant-leaf/versions/1


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

In [ ]:
import os
print(os.listdir(path))

['valid', 'README.dataset.txt', 'README.roboflow.txt', 'train']


In [ ]:
train_dir = path
train_dir = os.path.join(path, "Train")
train_dir = os.path.join(path, "data")
print(os.listdir(path))

['valid', 'README.dataset.txt', 'README.roboflow.txt', 'train']


In [ ]:
train_dir = os.path.join(path, "Train")   # adjust if needed
val_dir = os.path.join(path, "Validation")  # ya same train split kar lenge

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os

train_dir = path   # 👈 safest (jab tak confirm na ho)

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

val_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=(128,128),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)

Found 6965 images belonging to 2 classes.
Found 1741 images belonging to 2 classes.


In [ ]:
model = Sequential()

model.add(Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(128, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

model.add(Flatten())

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))

model.add(Dense(train_data.num_classes, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 327s 1s/step - accuracy: 0.9210 - loss: 0.3030 - val_accuracy: 0.9213 - val_loss: 0.2732
Epoch 2/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 372s 2s/step - accuracy: 0.9212 - loss: 0.2885 - val_accuracy: 0.9213 - val_loss: 0.2715
Epoch 3/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 326s 1s/step - accuracy: 0.9212 - loss: 0.2829 - val_accuracy: 0.9213 - val_loss: 0.2733
Epoch 4/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 321s 1s/step - accuracy: 0.9212 - loss: 0.2751 - val_accuracy: 0.9213 - val_loss: 0.2725
Epoch 5/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 327s 2s/step - accuracy: 0.9212 - loss: 0.2739 - val_accuracy: 0.9213 - val_loss: 0.2757
Epoch 6/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 323s 1s/step - accuracy: 0.9212 - loss: 0.2671 - val_accuracy: 0.9213 - val_loss: 0.2794
Epoch 7/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 325s 1s/step - accuracy: 0.9212 - loss: 0.2630 - val_accuracy: 0.9213 - val_loss: 0.2753
Epoch 8/10
218/218 ━━━━━━━━━━━━━━━━━━━━ 322s 1s/step - accuracy: 0.9212 - loss: 0.2632 - val_accu

In [ ]:
loss, acc = model.evaluate(val_data)
print("Validation Accuracy:", acc)

55/55 ━━━━━━━━━━━━━━━━━━━━ 38s 684ms/step - accuracy: 0.9207 - loss: 0.2891
Validation Accuracy: 0.9207351803779602


In [ ]:
model.save("plant_disease_model.h5")

In [ ]:
import os

# kisi ek class folder ka path lo
class_folders = os.listdir(path)
first_class = os.path.join(path, class_folders[0])

# us class ke andar se ek image
img_name = os.listdir(first_class)[0]
img_path = os.path.join(first_class, img_name)

print("Using image:", img_path)

Using image: /root/.cache/kagglehub/datasets/hriitk2026/plant-leaf/versions/1/valid/20_Rice_Healthy_val_jpg.rf.977e436e498fce6ed3e4d9013cd08ab2.jpg


In [ ]:
from tensorflow.keras.preprocessing import image
import numpy as np

img = image.load_img(img_path, target_size=(128,128))
img_array = image.img_to_array(img) / 255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)
print("Prediction:", prediction)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
Prediction: [[0.93499213 0.06500786]]
